# 1. Simulación de datos sintéticos (FinanceAI)

**1. Simulación** > 2. EDA > 3. Entrenamiento

### Propósito del cuaderno
Este cuaderno construye desde cero el universo de datos del proyecto: 1,800 usuarios y 864,000 transacciones anuales (aproximadamente 40 operaciones mensuales por usuario).

En lugar de asignar etiquetas aleatorias arbitrarias, el perfil financiero de cada usuario (endeudamiento y ahorro) se deduce matemáticamente de su historial completo de consumos. Esto genera una correlación lógica estricta para que los algoritmos de Machine Learning aprendan patrones de comportamiento financiero realistas.

In [1]:
import pandas as pd
import numpy as np
import os
import json
from faker import Faker

# ---------------------------------------------------------
# Reproducibilidad experimental
# ---------------------------------------------------------
# Fijamos una semilla (seed) global para que NumPy y Faker inicien
# sus generadores pseudoaleatorios en el mismo punto matemático.
# Esto asegura que cualquier ejecución futura produzca exactamente
# el mismo conjunto de datos y resultados.
SEED = 42
np.random.seed(SEED)
Faker.seed(SEED)
fake = Faker('es_ES')

## 1. Generación de la población de usuarios

### ¿Qué buscamos?
Modelar una base de 1,800 clientes con nombres sintéticos y salarios mensuales verosímiles.

### Criterio técnico y distribución estadística:
* **Distribución triangular (`np.random.triangular`):** En economías reales, la distribución de ingresos no es plana (uniforme) ni simétrica pura (normal). Usamos una distribución con un piso de $1,500, un pico frecuente (moda) de $1,800 y un techo de $5,000.
* **Control de valores atípicos (outliers):** Este rango evita tanto sueldos negativos como valores extremos de personas multimillonarias, los cuales distorsionarían las proporciones de endeudamiento y capacidad de ahorro en los cálculos posteriores.

### Señales de control (Green Flags):
* El mínimo de ingresos debe ser exactamente $\ge 1,500$ y el máximo $\le 5,000$.
* La mediana debe situarse entre $2,200$ y $2,600$, reflejando una asimetría realista.

In [2]:
n_usuarios = 1800

# Generación de nombres ficticios en español para preservar privacidad
nombres = [fake.first_name() for _ in range(n_usuarios)]

# Generación de salarios mensuales con distribución triangular (mínimo=1500, moda=1800, máximo=5000)
ingresos = np.round(np.random.triangular(1500, 1800, 5000, size=n_usuarios), 2)

# Construcción del DataFrame de usuarios
df_usuarios = pd.DataFrame({
    'id': range(1, n_usuarios + 1),
    'nombre': nombres,
    'ingreso_mensual': ingresos
})

# Verificación rápida de las dimensiones y las últimas filas generadas
print(f"Muestra de usuarios:\n{df_usuarios.tail().to_string(index=False)}")
print("- - -")
print(f"Estructura generada (filas, columnas): {df_usuarios.shape}")

Check:
  id    nombre  ingreso_mensual
1796 Estefanía          1798.38
1797   Soledad          3217.54
1798     Chelo          1775.12
1799     Nacio          1773.53
1800   Yolanda          1612.76
- - -
Estructura generada (instances, atributo): (1800, 3)


## 2. Generación masiva de transacciones (864,000 registros)

### ¿Qué buscamos?
Simular el historial de consumo anual de los 1,800 usuarios (~40 transacciones mensuales por persona), respetando leyes microeconómicas y diversidad lingüística bancaria.

### Criterios técnicos de simulación:
1. **Frecuencias humanas ponderadas (`prob_categorias`):** Ponderación probabilística donde consumos diarios (Alimentación 20%, Servicios 20%) superan a compras esporádicas (Electrodomésticos 5%, Inversión 2.5%).
2. **Subpoblaciones conductuales:**
   * **15% Derrochadores:** Mayor propensión al gasto en Ocio/Vestimenta y gastos fijos multiplicados por 1.8.
   * **15% Austeros:** Mayor propensión al ahorro/inversión y gastos fijos reducidos al 30%.
3. **Índice de poder adquisitivo y elasticidad (`dict_indices`):** Los gastos elásticos (Ocio, Vivienda, Tecnología) escalan en función del salario del usuario, mientras que los inelásticos (Servicios) se mantienen estables.
4. **Montos con distribución triangular:** Gastos fijos con moda al 50% del rango (estabilidad) y gastos variables con moda al 25% (frecuencia de tickets pequeños).
5. **Perturbaciones léxicas para NLP:** Prefijos reales de pasarelas de pago (`t.deb`, `fac`, `compra`) y errores tipográficos para evitar que el modelo NLP memorice palabras perfectas.

### Señales de control (Green Flags):
* `Alimentacion` y `Servicios` deben liderar el recuento de transacciones; `Inversion` debe ser la de menor volumen.
* Los montos generados deben ser positivos y correlacionarse con el nivel de ingresos del usuario.

In [3]:
n_transacciones = 864000

# ---------------------------------------------------------
# 1. Diccionario de conceptos bancarios por categoría
# ---------------------------------------------------------
# Contiene más de 180 variantes de descripciones reales por cada una
# de las 10 categorías acordadas para el MVP.
diccionario_conceptos = {
    'Alimentacion': [
        'deb proveeduria', 'fac cencosud', 'transf super', 'tarjeta granja', 'deb panaderia',
        'deb la anonima', 'supermercado', 'compra kiosco', 't.deb fiambreria', 'fac pescaderia',
        'deb polleria', 'cuota disco', 'debito dietetica', 'pago granja', 'transf verduleria',
        'cuota coto', 't.cred cencosud', 't.cred jumbo', 'tarj changomas', 't.cred vea',
        'debito proveeduria', 'pago super', 't.deb toledo', 'debito walmart', 'tarjeta makro',
        'fac walmart', 't.deb polleria', 't.deb supermercado', 'fac kiosco', 'debito carniceria',
        't.cred toledo', 'pago almacen', 'compra fiambreria', 'compra walmart', 'pago jumbo',
        'deb minimercado', 'transf dia', 'tarjeta supermercado', 't.deb minimercado', 'carrefour',
        't.deb vital', 'cuota panaderia', 'pago makro', 'transf walmart', 'carniceria',
        'deb makro', 'tarjeta verduleria', 'debito panaderia', 'pago verduleria', 'debito jumbo',
        'tarj dia', 'tarjeta super', 'cuota vea', 'fac panaderia', 'maxikiosco',
        'tarj carrefour', 'fac jumbo', 'changomas', 'pago vea', 'compra changomas',
        'compra proveeduria', 'debito super', 'tarj dietetica', 'cuota vital', 't.cred proveeduria',
        'pago la anonima', 'pago maxikiosco', 'deb dietetica', 'compra polleria', 'debito granja',
        'fac granja', 'cuota verduleria', 'pago fiambreria', 'compra super', 'transf almacen',
        'compra supermercado', 't.cred la anonima', 't.cred carrefour', 'tarj almacen', 'transf proveeduria',
        'cuota carrefour', 'debito makro', 'deb verduleria', 'transf jumbo', 'tarjeta panaderia',
        't.cred minimercado', 'fac changomas', 'fac coto', 'deb toledo', 'compra dia',
        'minimercado', 'pago minimercado', 't.cred polleria', 't.deb carrefour', 'kiosco',
        'cuota pescaderia', 'tarj vital', 'tarj pescaderia', 'polleria', 'debito cencosud',
        'fac minimercado', 't.deb panaderia', 't.deb dia', 'fac carniceria', 'transf fiambreria',
        'tarjeta disco', 'transf disco', 'tarj panaderia', 't.cred supermercado', 't.deb carniceria',
        'deb carrefour', 'cuota toledo', 't.cred makro', 'cuota supermercado', 'tarjeta carrefour',
        't.deb kiosco', 'compra verduleria', 't.deb changomas', 'deb cencosud', 'pago polleria',
        't.deb maxikiosco', 'transf carniceria', 'compra maxikiosco', 'verduleria', 'coto',
        'tarj fiambreria', 'cuota maxikiosco', 't.deb granja', 'tarj polleria', 'compra panaderia',
        'tarj maxikiosco', 't.cred granja', 'transf toledo', 'tarjeta pescaderia', 'pago vital',
        'cuota carniceria', 'tarjeta fiambreria', 't.cred maxikiosco', 'cuota dia', 'pescaderia',
        'cuota super', 'transf supermercado', 'pago cencosud', 'compra dietetica', 'pago walmart',
        'debito vital', 'deb fiambreria', 'la anonima', 't.deb jumbo', 'cuota almacen',
        'tarj supermercado', 'tarjeta maxikiosco', 'deb pescaderia', 't.cred kiosco', 't.cred dia',
        'vea', 'compra carrefour', 'debito disco', 'fac almacen', 'deb carniceria',
        'jumbo', 'cuota granja', 'compra jumbo', 'tarj vea', 'compra pescaderia',
        'tarjeta carniceria', 'fac makro', 't.cred disco', 'debito almacen', 'pago pescaderia',
        'tarj makro', 'tarjeta toledo', 'debito coto', 'pago carrefour', 'compra toledo',
        't.cred verduleria', 'tarj verduleria', 'tarj carniceria', 'tarjeta vea', 'cuota dietetica'
    ],
    'Educacion': [
        'compra escuela', 'pago colegio', 'deb curso', 'transf udemy', 'debito coursera',
        'tarjeta coursera', 't.cred colegio', 'deb domestika', 't.cred domestika', 'cuota udemy',
        'fac libreria', 't.cred facultad', 'debito instituto', 'compra coursera', 'transf tutor',
        't.deb profesor', 'tutor', 't.deb curso', 'transf instituto', 'cuota facultad',
        'compra english', 'pago domestika', 't.deb academia', 't.deb edx', 'tarj academia',
        't.cred jardin', 'pago english', 'debito curso', 'tarjeta facultad', 'compra matematica',
        'tarjeta profesor', 'fac profesor', 'pago matematica', 'libreria', 'tarj platzi',
        'fac platzi', 'compra jardin', 'deb escuela', 'domestika', 't.deb facultad',
        'fac fotocopias', 'pago edx', 'cuota papeleria', 'tarj papeleria', 't.deb tutor',
        'transf matematica', 'deb centro de idiomas', 'tarjeta papeleria', 'tarjeta domestika', 'tarj udemy',
        'tarjeta instituto', 'cuota tutor', 'instituto', 'coderhouse', 'cuota fotocopias',
        'cuota academia', 'compra guarderia', 'fac academia', 'debito facultad', 'tarjeta tutor',
        'fac colegio', 'debito libreria', 'tarj matematica', 't.deb coderhouse', 'tarj coursera',
        't.cred libreria', 't.cred guarderia', 'fac domestika', 'fac curso', 'jardin',
        'compra colegio', 'matematica', 'debito platzi', 'centro de idiomas', 'transf jardin',
        'debito centro de idiomas', 'cuota domestika', 'compra profesor', 'cuota escuela', 'compra academia',
        'debito escuela', 't.cred platzi', 'english', 'academia', 't.deb escuela',
        'transf universidad', 'compra tutor', 'pago escuela', 'guarderia', 'compra centro de idiomas',
        'pago academia', 'cuota platzi', 'tarj facultad', 'deb coderhouse', 'cuota profesor',
        'transf fotocopias', 't.deb domestika', 'pago profesor', 'deb coursera', 'transf libreria',
        'tarjeta jardin', 'tarj english', 'transf profesor', 'transf platzi', 'deb english',
        't.deb colegio', 'tarjeta platzi', 'udemy', 'deb matematica', 'compra papeleria',
        'tarjeta universidad', 'compra fotocopias', 'debito profesor', 'fac udemy', 'debito tutor',
        'compra domestika', 'pago coderhouse', 'fac jardin', 'tarj edx', 'pago curso',
        'deb academia', 'transf coderhouse', 'compra edx', 'pago fotocopias', 'deb jardin',
        'fac papeleria', 't.cred matematica', 'pago guarderia', 'compra instituto', 'pago udemy',
        'fac instituto', 't.cred centro de idiomas', 'escuela', 'pago jardin', 'deb edx',
        'tarjeta colegio', 'transf domestika', 't.deb instituto', 'tarjeta udemy', 'edx',
        'cuota libreria', 'colegio', 'platzi', 'deb fotocopias', 'deb guarderia',
        'debito english', 'tarjeta coderhouse', 'tarj colegio', 'tarj escuela', 'pago platzi',
        'tarj centro de idiomas', 't.deb fotocopias', 'deb papeleria', 'tarjeta fotocopias', 'fotocopias',
        'tarj domestika', 'tarjeta matematica', 't.deb matematica', 'cuota guarderia', 'pago facultad',
        'tarjeta academia', 'fac guarderia', 'pago universidad', 'deb facultad', 'deb platzi',
        'cuota english', 'transf escuela', 'papeleria', 'tarj fotocopias', 't.deb coursera',
        't.cred edx', 'cuota instituto', 'fac tutor', 'cuota curso', 'universidad',
        'debito domestika', 'cuota coderhouse', 't.deb udemy', 't.cred profesor', 'fac centro de idiomas'
    ],
    'Electrodomesticos': [
        'pago electrodomesticos', 'deb mercadolibre', 'tarjeta garbarino', 'cuota iper', 'pago philips',
        'tarj pc', 'debito casa del audio', 'tarjeta electro', 'compra electrodomesticos', 't.cred cetrogar',
        'debito garbarino', 'pago naldo', 'pago tienda mia', 'fac hendel', 'compra whirlpool',
        't.cred mercadolibre', 'deb casa del audio', 'fac tecnologia', 'compra tienda mia', 't.deb compumundo',
        'cuota compumundo', 'cetrogar', 'transf cetrogar', 'debito electrodomesticos', 't.deb electro',
        'tarjeta meli', 'fac compumundo', 'debito electro', 'tarjeta compumundo', 'debito cetrogar',
        'cuota megatone', 'cuota lg', 't.cred electrodomesticos', 'pago lg', 't.deb hendel',
        't.deb cetrogar', 'compra tecnologia', 'pago megatone', 'iper', 'pc',
        'macstation', 'fac cetrogar', 'tarj mercadolibre', 'compra compumundo', 'fac notebook',
        'tarj tecnologia', 'deb samsung', 'fac tienda mia', 'transf musimundo', 'cuota casa del audio',
        't.deb megatone', 'cuota mercadolibre', 'electrodomesticos', 'deb electrodomesticos', 'transf hendel',
        't.cred philips', 'debito tienda mia', 'transf notebook', 'debito hendel', 'debito notebook',
        'cuota garbarino', 'tarj macstation', 'compra lg', 'naldo', 'deb whirlpool',
        'pago notebook', 'tarj electro', 'deb hendel', 'tarj tienda mia', 'fac whirlpool',
        'cuota electrodomesticos', 'notebook', 'pago fravega', 't.deb electrodomesticos', 'fac meli',
        't.cred tecnologia', 't.deb naldo', 'tarjeta megatone', 'transf meli', 'tarj casa del audio',
        'debito lg', 'deb musimundo', 'fac fravega', 'transf iper', 'cuota tienda mia',
        'fac electro', 'debito compumundo', 'tarjeta musimundo', 'megatone', 'tarjeta mercadolibre',
        'pago macstation', 'compra pc', 'debito musimundo', 'debito mercadolibre', 'pago iper',
        'deb naldo', 'mercadolibre', 'transf sony', 't.cred compumundo', 't.deb tecnologia',
        'fac apple', 'cuota musimundo', 'samsung', 'transf electro', 'transf casa del audio',
        'tarjeta pc', 'sony', 't.deb whirlpool', 'cuota notebook', 't.cred hendel',
        'tarjeta electrodomesticos', 'pago sony', 'casa del audio', 'deb cetrogar', 'meli',
        'cuota tecnologia', 'pago mercadolibre', 'cuota samsung', 't.deb macstation', 'fac casa del audio',
        't.deb musimundo', 'tarjeta cetrogar', 'debito macstation', 't.cred musimundo', 'whirlpool',
        'tarjeta macstation', 'deb megatone', 'debito whirlpool', 't.deb casa del audio', 'pago pc',
        'tarjeta lg', 'compra naldo', 't.deb iper', 'compra casa del audio', 't.cred fravega',
        'debito sony', 't.deb tienda mia', 'compra notebook', 'tarjeta fravega', 't.cred macstation',
        'fac naldo', 'transf compumundo', 't.deb pc', 'tarjeta whirlpool', 't.deb fravega',
        'compra musimundo', 'deb lg', 'tarjeta philips', 'fac philips', 'deb sony',
        't.deb samsung', 'pago electro', 't.deb garbarino', 'cuota whirlpool', 'fac electrodomesticos',
        'debito pc', 'tarj lg', 'tarj philips', 't.deb philips', 'tarj musimundo',
        'compra megatone', 'tarjeta casa del audio', 'pago cetrogar', 'electro', 'debito philips',
        'transf pc', 'debito tecnologia', 'cuota meli', 't.cred pc', 'fac macstation',
        't.cred lg', 't.cred tienda mia', 'tarj fravega', 't.cred garbarino', 'compumundo',
        'tarj cetrogar', 'transf macstation', 't.deb lg', 'cuota sony', 'deb apple'
    ],
    'Inversion': [
        'cuota mep', 't.cred binance', 'debito binance', 'pago plazo fijo', 'fac buenbit',
        't.cred acciones', 'deb mep', 'cuota iol', 'compra buenbit', 't.cred santander',
        'cuota broker', 'tarjeta invertironline', 'pago acciones', 'cedear', 'cripto',
        'tarj bbva', 't.cred bull market', 't.deb buenbit', 'fac satoshitango', 'tarjeta lemon',
        'compra cedear', 'debito bull market', 'fac bull market', 'pago balanz', 't.deb portfolio persnl',
        'debito bonos', 'fac mep', 'bonos', 'cuota buenbit', 'tarjeta balanz',
        'deb binance', 'compra plazo fijo', 'plazo fijo', 'fac binance', 'acciones',
        't.deb invertironline', 't.cred banco galicia', 't.deb cocos capital', 'transf bonos', 'debito cocos capital',
        'deb compra dolar', 'compra santander', 'transf portfolio persnl', 'transf cocos capital', 'ripio',
        'cuota banco galicia', 'deb broker', 'pago macro', 'pago portfolio personal', 'transf balanz',
        'lemon', 't.cred mep', 'tarjeta portfolio persnl', 'fac fci', 't.deb broker',
        'pago portfolio persnl', 'transf broker', 't.cred compra dolar', 'tarj binance', 't.deb mep',
        'fac cripto', 'debito satoshitango', 'tarjeta bull market', 't.deb compra dolar', 'compra belo',
        'fac iol', 't.deb santander', 'cuota bonos', 'deb bbva', 'fac banco galicia',
        'compra compra dolar', 'tarj buenbit', 'debito belo', 'pago ripio', 'transf compra dolar',
        'belo', 'banco galicia', 'compra lemon', 'compra ripio', 'debito compra dolar',
        't.deb bonos', 't.cred broker', 'portfolio personal', 'fac cedear', 'tarjeta portfolio personal',
        'tarjeta ppi', 'cuota satoshitango', 'fac macro', 'tarj compra dolar', 'fac santander',
        'tarj bull market', 'fac invertironline', 'santander', 'tarj cedear', 'pago lemon',
        'cuota compra dolar', 'tarj lemon', 'deb cocos capital', 'debito ppi', 'pago banco galicia',
        'tarjeta buenbit', 'compra banco', 'debito bbva', 'compra balanz', 'deb bonos',
        'tarj fci', 'compra portfolio personal', 't.cred iol', 'tarj portfolio personal', 'deb portfolio personal',
        'tarjeta compra dolar', 'deb invertironline', 'tarj macro', 't.cred banco', 'debito banco',
        'tarj cripto', 'fac banco', 'portfolio persnl', 'compra macro', 'deb macro',
        'compra cripto', 'tarj acciones', 't.cred cocos capital', 'transf binance', 'fac ppi',
        'compra ppi', 'pago cripto', 'tarjeta plazo fijo', 't.deb balanz', 'cuota ripio',
        'debito balanz', 'cuota invertironline', 'transf banco galicia', 'cuota fci', 'fac balanz',
        'pago banco', 'tarj plazo fijo', 'pago ppi', 't.cred portfolio persnl', 'deb santander',
        'transf santander', 't.cred cripto', 'debito iol', 'tarjeta santander', 'transf cripto',
        'deb acciones', 'tarj iol', 'macro', 'pago bonos', 'banco',
        'debito portfolio persnl', 'deb balanz', 'pago belo', 'fac acciones', 'fac bonos',
        'bbva', 'deb plazo fijo', 'debito acciones', 'compra mep', 'compra acciones',
        't.cred lemon', 'compra broker', 'tarj ppi', 'transf plazo fijo', 'deb iol',
        'cuota belo', 't.deb iol', 'buenbit', 't.deb banco', 'pago cedear',
        't.deb cedear', 'debito cripto', 'transf lemon', 'cuota acciones', 't.deb ripio',
        't.deb belo', 'tarj ripio', 'pago santander', 'fac ripio', 'deb lemon'
    ],
    'Ocio': [
        'debito museo', 'tarjeta amazon prime', 'gimnasio', 't.deb boliche', 't.deb hoyts',
        'tarjeta bar', 'debito youtube', 'tarjeta hoyts', 'tarjeta netflx', 'cuota recital',
        'bar', 'pago netflx', 'deb cerveceria', 'cuota restaurante', 'pago boliche',
        'debito gimnasio', 't.deb spotify', 't.deb pub', 'tarj amazon prime', 'pago starbucks',
        'pago showcase', 'deb cine', 'tarjeta museo', 'tarjeta youtube', 't.cred burger king',
        'transf juego', 'transf cafe', 'pago hoyts', 'debito sptfy', 't.cred showcase',
        'tarj showcase', 'deb mcdonalds', 'tarjeta nintendo', 'compra hoyts', 'tarjeta mostaza',
        'tarjeta passline', 'pago hbo', 'deb amazon prime', 't.cred juego', 'tarjeta juego',
        'transf steam', 'tarjeta playstation', 'pago passline', 'fac netflix', 'deb boliche',
        'deb youtube', 'cerveceria', 'compra pub', 'pago restaurante', 'cuota cine',
        'tarjeta showcase', 't.deb gimnasio', 'tarj pub', 'transf boliche', 'mcdonalds',
        'tarj cinemark', 'compra youtube', 'cuota netflix', 'tarj teatro', 'compra nintendo',
        't.cred pub', 'debito cinemark', 't.deb restaurante', 'debito entradas', 't.cred restaurante',
        't.cred netflix', 'cuota spotify', 'cuota playstation', 't.deb showcase', 't.cred playstation',
        'entradas', 'cuota disney', 'tarj burger king', 't.cred cerveceria', 'tarj passline',
        'fac cinemark', 't.deb mostaza', 'debito mcdonalds', 'pago cafe', 'debito netflix',
        'compra steam', 'tarj ticketek', 'compra cine', 'compra xbox', 'pago recital',
        'cuota steam', 'transf mostaza', 'fac disney', 'compra entradas', 'cuota hoyts',
        'tarj disney', 'fac youtube', 'debito playstation', 't.deb xbox', 'compra boliche',
        'compra juego', 'transf amazon prime', 'deb passline', 'compra cafe', 'cine',
        'compra disney', 'transf club', 'debito disney', 't.cred youtube', 'deb pub',
        't.deb cine', 'debito club', 'debito boliche', 'debito ticketek', 'tarjeta gimnasio',
        'tarj netflix', 'playstation', 'tarj restaurante', 'compra netflx', 'tarjeta cerveceria',
        't.cred steam', 'pago nintendo', 'fac netflx', 'fac recital', 't.deb amazon prime',
        'deb disney', 'burger king', 'cuota starbucks', 'transf passline', 'tarj juego',
        'compra ticketek', 'netflx', 'fac entradas', 'fac burger king', 't.deb ticketek',
        'transf netflix', 't.cred bar', 'compra passline', 'tarjeta spotify', 'transf hbo',
        't.deb starbucks', 'tarj hbo', 'tarjeta hbo', 'pago amazon prime', 'fac teatro',
        'deb ticketek', 't.cred gimnasio', 'cuota showcase', 't.deb recital', 't.cred cine',
        'netflix', 'pago cinemark', 'recital', 't.deb nintendo', 'transf playstation',
        'fac starbucks', 'pago museo', 'cuota nintendo', 'tarjeta xbox', 'steam',
        'tarjeta restaurante', 'cuota sptfy', 't.cred spotify', 't.cred cafe', 'compra bar',
        'deb netflix', 'cuota boliche', 'cuota bar', 'fac ticketek', 'tarj cine',
        't.cred museo', 't.deb hbo', 'deb restaurante', 'deb hbo', 'compra club',
        'deb gimnasio', 'tarj bar', 'teatro', 'fac club', 'pago gimnasio',
        't.cred amazon prime', 'deb nintendo', 'tarjeta netflix', 'compra amazon prime', 'pago youtube'
    ],
    'Salud': [
        'compra osde', 'compra laboratorio', 'deb analisis', 'clinica', 'cuota kinesiologo',
        'fac medicamentos', 'compra lentes', 'tarj clinica', 't.cred psicologo', 'cuota farmacia',
        'compra galeno', 'tarj osde', 'fac farmacia', 't.cred dr ahorro', 'fac hospital',
        'deb medicamentos', 'transf sanatorio', 'fac farmac', 't.cred prepagas', 'compra dentista',
        'debito kinesiologo', 'cuota psicologo', 'tarjeta clinica', 'tarj odontologo', 'transf prepagas',
        't.deb terapia', 'compra medife', 'tarjeta dr ahorro', 't.cred obra social', 't.deb galeno',
        'pago farmac', 'analisis', 'debito galeno', 'sanatorio', 'pago dr ahorro',
        'pago obra social', 'debito centro medico', 'centro medico', 'farmacia', 'transf medicamentos',
        'transf lentes', 'tarjeta osde', 'fac sancor salud', 'debito analisis', 't.deb medife',
        'tarjeta prepagas', 'debito farmacity', 'pago odontologo', 'fac analisis', 't.cred optica',
        'transf medife', 'debito osde', 'cuota centro medico', 'deb odontologo', 'fac sanatorio',
        'fac swiss medical', 'cuota sancor salud', 'tarjeta lentes', 'fac prepagas', 'transf swiss medical',
        'transf laboratorio', 't.cred farmacity', 'tarjeta laboratorio', 'tarj dr ahorro', 'pago medicamentos',
        'debito hospital', 'fac odontologo', 'tarj kinesiologo', 'compra farmac', 'cuota osde',
        'cuota medicamentos', 'tarjeta sanatorio', 't.cred galeno', 'tarj medife', 't.cred centro medico',
        'tarj terapia', 'cuota dentista', 't.cred medife', 'compra odontologo', 't.cred kinesiologo',
        'tarj centro medico', 'debito farmacia', 'pago farmacity', 'pago kinesiologo', 'compra obra social',
        'debito clinica', 'deb farmacia', 'dr ahorro', 'fac clinica', 'tarj analisis',
        't.cred lentes', 'fac centro medico', 'tarjeta medicamentos', 'pago dentista', 'fac psicologo',
        'cuota sanatorio', 'deb osde', 'cuota hospital', 'transf obra social', 'galeno',
        'deb prepagas', 'tarjeta swiss medical', 'tarj laboratorio', 'transf dr ahorro', 'debito dr ahorro',
        'pago centro medico', 'tarj farmacity', 't.deb dentista', 'deb terapia', 'transf analisis',
        'odontologo', 'transf dentista', 'deb centro medico', 'deb dr ahorro', 'deb sancor salud',
        'tarjeta farmac', 'pago terapia', 'debito optica', 'tarjeta dentista', 'debito medicamentos',
        'cuota farmacity', 'tarjeta medife', 't.deb obra social', 'tarj lentes', 'compra sanatorio',
        'deb psicologo', 'prepagas', 'deb kinesiologo', 't.deb osde', 'compra kinesiologo',
        'pago galeno', 'pago prepagas', 't.deb kinesiologo', 'pago clinica', 'pago sancor salud',
        'compra optica', 't.cred dentista', 'transf sancor salud', 'swiss medical', 't.cred medicamentos',
        't.cred terapia', 't.deb lentes', 'compra centro medico', 'transf psicologo', 'debito obra social',
        'tarjeta centro medico', 'transf clinica', 'tarjeta farmacia', 'tarj prepagas', 'tarjeta sancor salud',
        'tarj psicologo', 'compra swiss medical', 'transf centro medico', 'tarjeta odontologo', 'compra farmacity',
        'transf galeno', 'compra psicologo', 'tarj hospital', 'pago sanatorio', 'psicologo',
        'pago swiss medical', 'transf hospital', 'deb farmacity', 't.deb hospital', 'cuota obra social',
        'deb dentista', 'tarj farmacia', 't.cred odontologo', 'cuota lentes', 't.deb centro medico',
        'tarj obra social', 'kinesiologo', 'debito psicologo', 'fac dr ahorro', 't.deb analisis',
        't.deb sanatorio', 'tarj sanatorio', 'tarj medicamentos', 'compra hospital', 'deb sanatorio',
        'ibuprofeno', 'ibupirac'
    ],
    'Servicios': [
        't.cred telecentro', 'pago abl', 'tarjeta ecogas', 't.deb luz', 'cuota arba',
        'debito seg', 'cuota telecentro', 'cuota claro', 't.deb tuenti', 'debito rentas',
        'debito metrogas', 'pago luz', 'tarjeta internet', 'cuota personal', 'fac patente',
        'debito fibertel', 'fac gas', 't.cred agip', 'transf persnl', 'tarjeta flow',
        'deb aysa', 'pago afip', 'arba', 'ecogas', 'tarj agip',
        'tarjeta patente', 'tarjeta edenor', 't.cred gas', 'fac persnl', 'deb personal',
        'tarjeta rentas', 'deb fibertel', 'pago gas', 'pago cablevision', 'transf pago mis cuentas',
        'tarj edenor', 'seg', 'expensas', 'deb gas', 'tarjeta persnl',
        'cuota cablevision', 'fac tuenti', 'debito movist', 'pago aysa', 'telecentro',
        'compra patente', 'tarj edesur', 't.cred naturgy', 't.deb aysa', 'agua',
        'deb edesur', 'compra fibertel', 'compra flow', 'pago agua', 'tarjeta expensas',
        'compra seg', 'cuota movistar', 'tarj metrogas', 'debito gas', 'fac movistar',
        'tarj afip', 'fac movist', 't.cred flow', 'debito pago mis cuentas', 't.deb fibertel',
        'cablevision', 't.cred fibertel', 'transf metrogas', 'tarj internet', 'deb patente',
        'patente', 'tarj flow', 'cuota persnl', 'tarj patente', 't.deb telecentro',
        't.cred rentas', 't.cred metrogas', 'fac arba', 'tarjeta fibertel', 'deb arba',
        'compra personal', 't.cred luz', 't.deb ecogas', 'transf abl', 'pago rentas',
        'tarjeta abl', 'transf patente', 'tarj cablevision', 't.cred abl', 'deb directv',
        'compra seguro', 'claro', 't.deb flow', 'compra arba', 'transf edesur',
        'tarjeta seguro', 'movistar', 't.cred edesur', 'fac expensas', 'transf luz',
        'personal', 'tarjeta edesur', 'compra agua', 't.deb edesur', 't.cred persnl',
        'tarjeta cablevision', 'pago seguro', 't.cred seguro', 'pago patente', 'compra telecentro',
        'transf claro', 'transf tuenti', 'cuota aysa', 't.cred seg', 'transf directv',
        'fac personal', 'debito edenor', 'tarjeta naturgy', 'debito movistar', 'cuota camuzzi',
        'afip', 't.cred expensas', 'tarjeta telecentro', 'transf personal', 'fac directv',
        'cuota agip', 'edesur', 'deb tuenti', 'debito personal', 'debito patente',
        'camuzzi', 't.deb agip', 'debito directv', 't.deb claro', 'pago movist',
        't.cred patente', 'compra movist', 'transf naturgy', 'cuota agua', 'tarjeta directv',
        'transf flow', 'debito agua', 'compra gas', 't.cred movistar', 'transf seg',
        'transf afip', 't.cred camuzzi', 'deb telecentro', 't.cred afip', 't.cred aysa',
        'compra persnl', 'tarjeta personal', 'cuota afip', 'tarj gas', 'tarj tuenti',
        'deb abl', 'tarjeta movist', 'fac edesur', 't.deb movistar', 'compra naturgy',
        'deb flow', 'deb camuzzi', 'tarjeta aysa', 'deb metrogas', 't.deb metrogas',
        'compra edesur', 'debito telecentro', 'compra tuenti', 'transf movist', 'deb naturgy',
        't.deb agua', 'compra rentas', 'tarj rentas', 'tarjeta metrogas', 'cuota gas',
        'cuota directv', 't.cred edenor', 'fac afip', 'deb agip', 't.cred movist'
    ],
    'Transporte': [
        'pago ubr', 'tarj jetsmart', 'tarj remis', 'tren', 'tarjeta estacionamiento',
        'compra shell', 'tarjeta puma', 't.cred despegar', 'tarj tren', 'tarjeta uber',
        'compra flybondi', 'fac ypf', 'compra tren', 'fac despegar', 'pago cabify',
        'cuota colectivo', 'cuota cabify', 'tarj axion', 'cuota didi', 'compra despegar',
        't.deb flybondi', 't.deb remis', 'fac pasaje', 't.deb didi', 'compra sube',
        'deb tren', 'tarjeta pasaje', 'compra cabify', 'transf peaje', 'fac estacionamiento',
        'compra latam', 'tarjeta flybondi', 'tarj uber', 'pago remis', 'tarj turismocity',
        'cuota boleto', 'cuota puma', 'transf shell', 'fac jetsmart', 'transf estacionamiento',
        'fac puma', 't.cred nafta', 'transf flybondi', 'pago despegar', 'transf beat',
        'tarjeta boleto', 'colectivo', 'tarjeta beat', 'pago puma', 'deb uber',
        'pago turismocity', 'compra ypf', 'deb shell', 'aerolineas', 't.cred sube',
        't.deb nafta', 'tarj subte', 'despegar', 'debito peaje', 'tarjeta tren',
        'cuota uber', 'deb boleto', 'tarjeta turismocity', 'pago boleto', 'transf puma',
        'beat', 'transf nafta', 't.cred ypf', 'cuota subte', 't.deb puma',
        'fac axion', 'deb subte', 'debito despegar', 'compra jetsmart', 'transf latam',
        'transf combustible', 'transf taxi', 'tarjeta nafta', 't.deb pasaje', 'compra cbfy',
        'tarj combustible', 'cbfy', 'cuota jetsmart', 't.cred flybondi', 'pago colectivo',
        'pago uber', 'cuota cbfy', 'tarjeta axion', 'debito tren', 'transf tren',
        't.cred estac', 'deb cabify', 'deb beat', 'compra telepase', 'transf sube',
        'pago axion', 'fac boleto', 'pago shell', 'fac didi', 'tarj telepase',
        'transf aerolineas', 't.cred uber', 'tarj ypf', 't.cred axion', 'pago estacionamiento',
        'tarjeta sube', 't.cred ubr', 't.cred pasaje', 'taxi', 'deb puma',
        'pago jetsmart', 'cuota pasaje', 't.deb turismocity', 'deb remis', 'tarj colectivo',
        'deb estacionamiento', 'deb colectivo', 'debito aerolineas', 'pago estac', 'fac colectivo',
        'tarjeta colectivo', 't.cred puma', 'pago tren', 'telepase', 'debito cabify',
        'boleto', 't.deb tren', 'cuota peaje', 'fac flybondi', 'uber',
        'transf jetsmart', 't.deb latam', 'deb taxi', 'compra combustible', 'compra didi',
        'debito didi', 'latam', 'debito combustible', 'ypf', 'tarj flybondi',
        't.deb cabify', 'fac nafta', 'deb despegar', 'compra boleto', 'peaje',
        'cuota combustible', 'didi', 't.deb peaje', 'debito axion', 'debito cbfy',
        'deb jetsmart', 'compra turismocity', 'debito jetsmart', 'cuota estacionamiento', 'tarjeta latam',
        'deb flybondi', 'debito flybondi', 'pago telepase', 'pago nafta', 'compra estac',
        'cuota shell', 'deb latam', 'transf despegar', 'pago beat', 'cuota nafta',
        'debito ypf', 'fac uber', 'pago taxi', 'cuota axion', 'jetsmart',
        'debito estacionamiento', 'fac ubr', 'pago latam', 'cuota remis', 't.deb ypf',
        'pago aerolineas', 'tarj pasaje', 'fac tren', 'tarj sube', 'pago sube'
    ],
    'Vestimenta': [
        'pago abasto', 'tarj zapateria', 'fac outlet', 'tarjeta falabella', 't.deb puma',
        'transf gucci', 't.deb local', 'compra lenceria', 't.cred moda', 'pago falabella',
        'debito falabella', 'tarjeta prada', 'cuota outlet', 't.deb zara', 'pago lenceria',
        'pago gucci', 'fac lenceria', 'puma', 'fac zara', 'debito zapatillas',
        'tarjeta local', 'tarj alto palermo', 'pago alto palermo', 'cuota nike', 'cuota moda',
        'alto palermo', 'tarjeta abasto', 'tarjeta indumentaria', 'pago moda', 'debito alto palermo',
        'abasto', 'debito unicenter', 'pago unicenter', 'tarjeta moda', 'compra zara',
        'cuota gucci', 'deb levis', 'compra abasto', 't.cred gucci', 'tarjeta levis',
        'transf outlet', 'deb puma', 'pago reebok', 't.cred nike', 'transf campera',
        'transf calzado', 'debito shopping', 'deb campera', 'moda', 'compra levis',
        'h&m', 'transf zara', 't.cred reebok', 't.cred campera', 'transf levis',
        'fac indumentaria', 'compra indumentaria', 'fac jeans', 'deb h&m', 'compra calzado',
        'tarj unicenter', 'transf zapateria', 'debito under armour', 'compra campera', 't.cred falabella',
        'pago h&m', 'pago zapateria', 'cuota prada', 'cuota levis', 't.deb falabella',
        'cuota unicenter', 'zara', 'pago campera', 'deb falabella', 'debito calzado',
        'tarj outlet', 'compra zapatillas', 'deb gucci', 'fac prada', 'tarjeta gucci',
        'cuota indumentaria', 'debito indumentaria', 't.deb h&m', 'debito abasto', 'compra puma',
        'tarjeta lenceria', 'tarj reebok', 'transf unicenter', 'debito lenceria', 'transf zapatillas',
        'deb local', 'transf local', 'fac reebok', 'transf nike', 'pago prada',
        'deb zapatillas', 'deb lenceria', 'cuota lenceria', 'compra falabella', 'compra moda',
        'cuota calzado', 'tarjeta campera', 'tarjeta ropa', 'calzado', 'tarj gucci',
        'transf prada', 'tarj jeans', 'adidas', 't.deb shopping', 't.cred h&m',
        'debito nike', 'compra unicenter', 'compra h&m', 'cuota local', 'tarjeta outlet',
        'debito local', 'tarjeta reebok', 't.cred alto palermo', 'tarj indumentaria', 'debito moda',
        't.cred indumentaria', 'local', 'tarj shopping', 'tarjeta alto palermo', 'deb prada',
        't.cred boutique', 'pago levis', 't.cred zara', 'pago jeans', 'tarj zapatillas',
        'fac alto palermo', 't.deb lenceria', 'compra outlet', 'tarjeta shopping', 'compra shopping',
        'cuota puma', 'deb jeans', 'tarjeta h&m', 't.deb boutique', 'zapatillas',
        'deb reebok', 'transf shopping', 'transf adidas', 't.cred adidas', 't.deb indumentaria',
        'compra ropa', 'cuota shopping', 'shopping', 'tarj nike', 'transf alto palermo',
        'tarj lenceria', 'tarj puma', 'transf indumentaria', 'campera', 'deb under armour',
        'compra jeans', 'cuota campera', 'deb outlet', 'transf h&m', 'debito reebok',
        't.cred shopping', 't.cred zapatillas', 't.cred abasto', 'gucci', 'transf lenceria',
        't.deb abasto', 'tarjeta puma', 'pago ropa', 'outlet', 'tarjeta adidas',
        'tarjeta calzado', 't.deb under armour', 'debito levis', 'fac falabella', 'fac moda',
        'cuota abasto', 'fac campera', 'tarjeta zapateria', 't.deb zapatillas', 't.cred calzado'
    ],
    'Vivienda': [
        'tarjeta construccion', 'construccion', 'pago electr', 't.cred arreglos', 'tarjeta alquiler',
        'mantenimiento', 'deb sodimac', 'tarj ferreteria', 'deb plomero', 'debito inmobiliaria',
        'transf materiales', 'pago muebleria', 'fac easy', 'tarjeta pintureria', 't.cred materiales',
        'compra ferret', 'electr', 'debito construccion', 't.deb ikea', 'debito gasista',
        'debito pintureria', 'cuota albañil', 'tarj albañil', 'transf electr', 'compra bazar',
        'tarj gasista', 'compra consorcio', 'plomero', 'inmobiliaria', 'deb hipoteca',
        'tarj sodimac', 'deb easy', 'tarjeta plomero', 'debito mantenimiento', 'cuota construccion',
        'ferret', 'compra pintureria', 'pintureria', 'cuota hipoteca', 'debito electricista',
        't.deb albañil', 't.cred hipoteca', 't.deb easy', 'compra materiales', 'cuota ikea',
        't.deb construccion', 't.deb deco', 'fac pintureria', 'transf sodimac', 'compra inmobiliaria',
        'cuota materiales', 'tarjeta arreglos', 'tarjeta bazar', 'cuota gasista', 'compra hipoteca',
        't.deb consorcio', 't.deb limpieza', 'debito cerrajero', 't.deb pintureria', 'pago hipoteca',
        'compra alquiler', 't.cred ikea', 'pago plomero', 'pago deco', 'fac expensas',
        'compra albañil', 'compra electr', 't.cred alquiler', 'cuota alquiler', 'fac hipoteca',
        't.deb electricista', 'ikea', 'limpieza', 't.deb sodimac', 'fac cerrajero',
        'fac materiales', 'deb expensas', 'transf ferreteria', 'pago cerrajero', 't.deb materiales',
        'debito consorcio', 'pago ikea', 'deco', 't.cred sodimac', 'compra mantenimiento',
        't.cred ferret', 'tarjeta albañil', 'deb electricista', 'debito deco', 'pago ferreteria',
        'debito arreglos', 'debito easy', 'transf electricista', 'debito ferret', 'debito bazar',
        'easy', 'ferreteria', 'deb construccion', 't.cred inmobiliaria', 'cuota electricista',
        't.cred gasista', 't.cred plomero', 'tarjeta gasista', 't.deb hipoteca', 'compra sodimac',
        'deb ikea', 'compra electricista', 'deb arreglos', 'pago albañil', 'tarj construccion',
        'compra arreglos', 'compra easy', 'pago ferret', 'debito limpieza', 'fac plomero',
        't.deb arreglos', 'compra limpieza', 't.cred muebleria', 'transf arreglos', 't.deb alquiler',
        'pago construccion', 'fac limpieza', 'cuota limpieza', 'cuota inmobiliaria', 'tarj muebleria',
        'debito expensas', 'transf deco', 'compra construccion', 'debito hipoteca', 'deb inmobiliaria',
        'tarjeta cerrajero', 'pago sodimac', 'debito electr', 't.cred electr', 'cuota cerrajero',
        'deb alquiler', 'tarj hipoteca', 'deb gasista', 'compra deco', 'cuota easy',
        't.cred consorcio', 'pago gasista', 't.deb mantenimiento', 'tarj plomero', 'fac ferret',
        'cuota electr', 'transf consorcio', 'fac bazar', 'tarj easy', 't.deb ferret',
        'compra muebleria', 'deb ferreteria', 'pago mantenimiento', 'tarjeta sodimac', 'transf hipoteca',
        'pago consorcio', 'debito ikea', 'fac arreglos', 't.deb electr', 'sodimac',
        'pago pintureria', 'tarjeta consorcio', 'tarj inmobiliaria', 'compra expensas', 'tarj expensas',
        'tarj arreglos', 't.cred construccion', 'debito albañil', 'arreglos', 'fac electr',
        'debito muebleria', 'fac albañil', 'tarjeta easy', 'tarj materiales', 'debito materiales',
        'materiales', 'hipoteca', 'pago inmobiliaria', 't.cred limpieza', 'tarj pintureria'
    ]
}

# Limites (base) de montos realistas - puede cambiar
rangos_montos = {
    'Alimentacion': (15, 100), 'Educacion': (30, 200), 'Electrodomesticos': (200, 400), 'Inversion': (100, 300),
    'Ocio': (20, 80), 'Salud': (40, 200), 'Servicios': (10, 80), 'Transporte': (3, 200),
    'Vestimenta': (10, 300), 'Vivienda': (100, 450)
}

categorias = list(diccionario_conceptos.keys()) # Seteando las categorías
usuario_ids = np.random.randint(1, n_usuarios + 1, size=n_transacciones) 

# Asignando categorias simulando comportamientos de gasto frecuentes
prob_categorias = [0.20, 0.05, 0.05, 0.025, 0.10, 0.05, 0.20, 0.125, 0.05, 0.15] # Se busca respetar el orden alfabético de categorías
# Clasifica (sortea) el lote de transacciones. Utiliza un set de probabilidades calibrado para intentar emular frecuencias reales, 
# asegurando que ir al supermercado salga sorteado muchas más veces que comprar un electrodoméstico.
selected_categories = np.random.choice(categorias, size=n_transacciones, p=prob_categorias) 

# 4. INYECCIÓN DE PISTAS: Alterar categorías para ciertos perfiles secretos
usuarios_derrochadores = set(df_usuarios.sample(frac=0.15)['id'])
restantes = df_usuarios[~df_usuarios['id'].isin(usuarios_derrochadores)]
usuarios_austeros = set(restantes.sample(frac=0.176)['id'])
for i in range(n_transacciones):
    uid = usuario_ids[i]
    c = selected_categories[i]
    if uid in usuarios_derrochadores and c not in ['Vivienda', 'Servicios'] and np.random.rand() < 0.07:
        selected_categories[i] = np.random.choice(['Ocio', 'Vestimenta'])
    elif uid in usuarios_austeros and c not in ['Vivienda', 'Servicios'] and np.random.rand() < 0.05:
        selected_categories[i] = 'Inversion'

fechas_random = pd.to_datetime('2025-01-01') + pd.to_timedelta(np.random.randint(0, 365, size=n_transacciones), unit='d') # Repartiendo fechas random a las transacciones. Cuidado con años bisiestos

descripciones = []
montos = []

# Novedad: Diccionario rápido para conocer el índice de poder adquisitivo de cada usuario
# Base: 1500 dólares de sueldo.
# Funcionalidad sugerida para ahorrar RAM y ahorrar tiempo al código
dict_indices = {row['id']: (row['ingreso_mensual'] / 1500) for _, row in df_usuarios.iterrows()} # Itera usuario por usuario, tomando el sueldo minimo como denominador para repartir multiplicadores float usando el id usuario como clave (clave:valor) 
categorias_elasticas = ['Vivienda', 'Educacion', 'Inversion', 'Ocio', 'Vestimenta', 'Electrodomesticos'] # definición de categorías elásticas

# PRUEBA - Crear tickets de compra. Tiene en cuenta el poder adquisitivo y las categorías elásticas
for user_id, cat in zip(usuario_ids, selected_categories):
    desc = np.random.choice(diccionario_conceptos[cat])
    
    # Inyección de ruido en texto (simulando errores de usuario o teclado)
    # Un 20% de las descripciones recibe prefijos "ruidosos" (ej. "compra ", "tarjeta ", "fac "). 
    #Un 10% de las descripciones sufre sustituciones de caracteres simulando errores de tipeo o teclado
    # Previene el overfitting en el modelo de procesamiento de lenguaje natural (NLP).
    if np.random.rand() < 0.20: 
        prefijos = ["pago ", "compra ", "tarjeta ", "fac ", ""]
        desc = str(np.random.choice(prefijos)) + desc
    if np.random.rand() < 0.10: 
        desc = desc.replace("a", "q", 1) if "a" in desc else desc.replace("e", "w", 1)

    min_m, max_m = rangos_montos[cat]
    idx_adquisitivo = dict_indices[user_id]
    
    # Factor de escalado dinámico según sueldo y si es cat elástica o no
    if cat in categorias_elasticas:
        max_m = max_m * (1 + (idx_adquisitivo - 1) * 0.5)
        min_m = min_m * (1 + (idx_adquisitivo - 1) * 0.2) # el minimo sube, pero más suave
    else:
        # Inelásticas (crecen muy poco)
        max_m = max_m * (1 + (idx_adquisitivo - 1) * 0.2)

    # 2. FACTOR HUMANO: Desconectar gastos fijos del sueldo para estos perfiles
    if cat in ['Vivienda', 'Servicios']:
        if user_id in usuarios_derrochadores:
            max_m *= 1.8
            min_m *= 1.8
        elif user_id in usuarios_austeros:
            max_m *= 0.3
            min_m *= 0.3


    
    # --- Distribución Triangular (Mejora Estadística) ---
    # Gastos fijos = Servicios, Educación, Vivienda
    # Consumo variable = Alimentación, Ocio, Vestimenta, etc.
    if cat in ['Servicios', 'Educacion', 'Vivienda']:
        moda_m = min_m + (max_m - min_m) * 0.5 # moda en mitad del rango (tienden a ser estables)
    else:
        moda_m = min_m + (max_m - min_m) * 0.25 # moda en primer cuartil. Suelen ser gastos pequeños
    monto = round(np.random.triangular(min_m, moda_m, max_m), 2) # genera un número aleatorio dentro del rango, concentrando densidad alrededor de la moda
    descripciones.append(desc)
    montos.append(monto)



# 10% de ruido de etiquetado en categorías para evitar que el algoritmo NLP sea 100% perfecto - DESACTIVADO
ruido_mask = np.random.rand(n_transacciones) < 0.10
categorias_ruido = np.random.choice(categorias, size=ruido_mask.sum())
selected_categories[ruido_mask] = categorias_ruido

df_transacciones = pd.DataFrame({
    'id': range(1, n_transacciones + 1),
    'usuario_id': usuario_ids,
    'descripcion': descripciones,
    'valor': montos,
    'categoria': selected_categories,
    'fecha': fechas_random.strftime('%Y-%m-%d')
})

print("Transacciones simuladas:", df_transacciones.shape)


Transacciones simuladas: (864000, 6)


In [4]:
pd.Series({k: len(v) for k, v in diccionario_conceptos.items()}).sort_values(ascending=False)

Salud                182
Alimentacion         180
Educacion            180
Electrodomesticos    180
Inversion            180
Ocio                 180
Servicios            180
Transporte           180
Vestimenta           180
Vivienda             180
dtype: int64

### 2.1 Revisión de transacciones


In [5]:
# Para ver cuántas transacciones promedio tiene cada usuario
tx_por_usuario = df_transacciones.groupby('usuario_id').size().reset_index(name='tx_anuales')
tx_por_usuario['tx_mensuales'] = tx_por_usuario['tx_anuales'] / 12.0

print("Transacciones por usuario:")
display(tx_por_usuario[['tx_anuales', 'tx_mensuales']].mean())

Transacciones por usuario:


tx_anuales      480.0
tx_mensuales     40.0
dtype: float64

## 3. Perfilado financiero (Reglas de negocio vectorizadas)

### ¿Qué buscamos?
Calcular objetivamente el perfil de salud financiera de cada usuario (`Saludable`, `En observación`, `En riesgo`) a partir de su historial real de consumos e ingresos.

### Criterios técnicos de cálculo:
1. **Nivel de endeudamiento:** Porcentaje del ingreso mensual comprometido en gastos fijos de supervivencia (Vivienda + Servicios).
   $$\text{Endeudamiento} = \left( \frac{\text{Gasto Fijo Mensual}}{\text{Ingreso Mensual}} \right) \times 100$$
2. **Frecuencia de ahorro:** Promedio mensual de compras en la categoría `Inversión` (`Ninguna`, `Baja`, `Media`, `Alta`).
3. **Matriz de perfilado (`np.select`):**
   * **En riesgo:** Endeudamiento $> 26.0\%$ (compromiso financiero crítico).
   * **Saludable:** Endeudamiento $\le 22.0\%$ y ahorro `Media` o `Alta`.
   * **En observación:** Zonas intermedias (endeudamiento entre $22\%$ y $26\%$, o nula capacidad de ahorro).

### Señales de control (Green Flags):
* La distribución final debe ser consistente con la economía bancaria (~75% Saludables, ~15% En observación, ~10% En riesgo).
* No deben existir valores nulos (`NaN`) tras el merge de transacciones.

> **Aislar los gastos fijos (Vivienda y Servicios):**
> El algoritmo busca todos los pagos de alquiler, luz o agua de un usuario y saca el promedio de cuánto paga al mes. Supongamos que el usuario "Juan" paga un promedio de $600 mensuales por estos conceptos. 

> **Ratio de endeudamiento:**
> Toma esos $600 y los divide por su sueldo. El cálculo arroja un 40%. Eso significa que Juan compromete el 40% de su sueldo solo para sobrevivir mes a mes.

> **Calcular el hábito de ahorro:**
> El algoritmo cuenta cuántos "tickets de compra" de Juan pertenecen a la categoría "Inversión" a lo largo del año. Si Juan tiene cero tickets de inversión, se le cataloga con Frecuencia de Ahorro "Ninguna".

---

##### **Armado del perfil financiero:** Cruza el endeudamiento con el ahorro de la siguiente manera: 
> * Si el endeudamiento de la persona es mayor al 26% de su sueldo, automáticamente se etiqueta como 'En Riesgo' (la persona está ahogada financieramente).
> * Si su endeudamiento es bajo (menor al 22%) Y además tiene un hábito de ahorro Medio o Alto, se corona como 'Saludable'.
> * Cualquier combinación intermedia o gris (ejemplo, tiene poco endeudamiento pero nunca invierte un centavo), cae en la categoría 'En Observación'.



* Se calculó el promedio mensual (mean()) del gasto en categorías fijas (Vivienda y Servicios).
* Se obtuvo el ratio de endeudamiento dividiendo dicho promedio por el ingreso mensual del usuario.
* Se contabilizó la frecuencia de transacciones en la categoría "Inversión" para definir la categoría de ahorro.
* Se aplicó np.select() para asignar el perfil financiero según umbrales de endeudamiento (22% y 26%) y frecuencia de ahorro.

In [6]:
# 1. Promedio mensual de gastos fijos (Vivienda y Servicios)
gastos_fijos = df_transacciones[df_transacciones['categoria'].isin(['Vivienda', 'Servicios'])]
gastos_fijos_promedio = gastos_fijos.groupby(['usuario_id', 'categoria'])['valor'].mean().unstack(fill_value=0)
if 'Vivienda' not in gastos_fijos_promedio.columns: gastos_fijos_promedio['Vivienda'] = 0
if 'Servicios' not in gastos_fijos_promedio.columns: gastos_fijos_promedio['Servicios'] = 0
gastos_fijos_promedio['gasto_fijo_total'] = gastos_fijos_promedio['Vivienda'] + gastos_fijos_promedio['Servicios']

# 2. Conteo de inversiones
inversiones = df_transacciones[df_transacciones['categoria'] == 'Inversion']
conteo_inversiones = inversiones.groupby('usuario_id').size().reset_index(name='n_inversion_anual')

# 3. Merge con usuarios
df_calc = df_usuarios[['id', 'ingreso_mensual']].copy()
df_calc = df_calc.merge(gastos_fijos_promedio[['gasto_fijo_total']], left_on='id', right_on='usuario_id', how='left').fillna(0)
df_calc = df_calc.merge(conteo_inversiones, left_on='id', right_on='usuario_id', how='left').fillna(0)

# 4. Cálculos vectorizados
df_calc['nivel_endeudamiento'] = np.round((df_calc['gasto_fijo_total'] / df_calc['ingreso_mensual']) * 100, 2)
df_calc['nivel_endeudamiento'] = df_calc['nivel_endeudamiento'].clip(5.0, 90.0)

df_calc['inversiones_mensuales'] = df_calc['n_inversion_anual'] / 12.0

# Asignar frecuencia de ahorro
condiciones_ahorro = [
    df_calc['inversiones_mensuales'] <= 0.75, # Ninguna # Ninguna: Menos de 3 inversiones AL AÑO (Aísla el ahorro esporádico)
    df_calc['inversiones_mensuales'] < 1.0,   # Baja: Hasta 1 inversión al mes
    df_calc['inversiones_mensuales'] < 1.25    # Media: Hasta 3 inversiones al mes
]
opciones_ahorro = ['Ninguna', 'Baja', 'Media']
df_calc['frecuencia_ahorro'] = np.select(condiciones_ahorro, opciones_ahorro, default='Alta')

# Asignar perfil financiero
condiciones_perfil = [
    df_calc['nivel_endeudamiento'] > 26.0,
    (df_calc['nivel_endeudamiento'] > 22.0) | (df_calc['frecuencia_ahorro'] == 'Ninguna'),
    (df_calc['nivel_endeudamiento'] <= 22.0) & (df_calc['frecuencia_ahorro'].isin(['Media', 'Alta']))
]
opciones_perfil = ['En riesgo', 'En observacion', 'Saludable']
df_calc['perfil_financiero'] = np.select(condiciones_perfil, opciones_perfil, default='En observacion')



# Integrar resultados al dataframe original
df_usuarios = pd.merge(df_usuarios, df_calc[['id', 'nivel_endeudamiento', 'frecuencia_ahorro', 'perfil_financiero']], on='id')

print("Perfiles consistentes calculados. Distribución:")
print(df_usuarios['perfil_financiero'].value_counts(normalize=True) * 100)




Perfiles consistentes calculados. Distribución:
perfil_financiero
Saludable         75.000000
En observacion    15.277778
En riesgo          9.722222
Name: proportion, dtype: float64


## 4. Estrategia de partición anti-fugas (Data Leakage Prevention)

### ¿Qué buscamos?
Garantizar que la evaluación de los modelos de Machine Learning refleje el rendimiento en producción real sin sesgos de sobreestimación.

### Criterios de partición dual:
* **Para Usuarios (Split Transversal / Cross-sectional):** División aleatoria agrupada (60% Train, 20% Val, 20% Test) para entrenar y evaluar el clasificador de perfiles sobre clientes no vistos.
* **Para Transacciones (Split Temporal / Out-of-Time):**
  * **Train (Enero a Agosto - ~66%):** Datos históricos del pasado para entrenar el modelo NLP.
  * **Validation (Septiembre a Octubre - ~17%):** Datos intermedios para calibrar hiperparámetros y umbrales de confianza.
  * **Test (Noviembre a Diciembre - ~17%):** El "futuro" sellado para la evaluación ciega final.

### ¿Por qué split temporal en transacciones?
En producción, un modelo clasifica gastos que ocurrirán mañana basándose en lo que aprendió ayer. Un split aleatorio mezclaría compras de fin de año en el entrenamiento, permitiendo que el modelo "haga trampa" conociendo el futuro.

In [7]:
# 1. Split Transversal Agrupado (Group Holdout Split) por ID de Usuario
splits_usr = np.random.choice(['train', 'val', 'test'], size=n_usuarios, p=[0.6, 0.2, 0.2])
df_usuarios['split'] = splits_usr

# 2. Split Temporal (Out-of-Time) para Transacciones
# Enero a Agosto (train), Sept-Oct (val), Nov-Dic (test)
meses = pd.to_datetime(df_transacciones['fecha']).dt.month
condiciones = [
    meses <= 8,
    meses.isin([9, 10]),
    meses >= 11
]
df_transacciones['split'] = np.select(condiciones, ['train', 'val', 'test'])

print("Distribución Transversal de Usuarios (Cross-sectional):")
print(df_usuarios['split'].value_counts(normalize=True) * 100)
print("\nDistribución Temporal de Transacciones (Out-of-Time):")
print(df_transacciones['split'].value_counts(normalize=True) * 100)

Distribución Transversal de Usuarios (Cross-sectional):
split
train    59.555556
test     20.444444
val      20.000000
Name: proportion, dtype: float64

Distribución Temporal de Transacciones (Out-of-Time):
split
train    66.494329
test     16.771065
val      16.734606
Name: proportion, dtype: float64


## 5. Exportación de semillas de datos (Data Science vs. Backend)

### Arquitectura de exportación:
1. **Archivos para Data Science (`usuarios.csv`, `transacciones.csv`):** Contienen la columna `split` (`train`, `val`, `test`) necesaria para el flujo de análisis exploratorio (EDA) y modelado.
2. **Archivos limpios para Backend (`usuarios_backend.csv`, `transacciones_backend.csv`):** Se elimina la columna `split` para que los scripts de migración (Flyway/Seeder) en Spring Boot y MySQL puedan poblar la base de datos sin errores de esquema.

In [8]:
os.makedirs('data', exist_ok=True)

# 1. Exportación Cruda (Uso Interno de Data Science)
# Estos archivos mantienen la columna "split" esencial para EDA y Modelado.
df_usuarios.to_csv('data/usuarios.csv', index=False)
df_transacciones.to_csv('data/transacciones.csv', index=False)
df_usuarios.to_json('data/usuarios.json', orient='records', indent=2, force_ascii=False)
df_transacciones.to_json('data/transacciones.json', orient='records', indent=2, force_ascii=False)

# 2. Exportación Limpia (Uso Externo para Backend)
# Se remueve "split" para evitar problemas de compatibilidad en Java.
df_usuarios.drop(columns=['split']).to_csv('data/usuarios_backend.csv', index=False)
df_transacciones.drop(columns=['split']).to_csv('data/transacciones_backend.csv', index=False)
df_usuarios.drop(columns=['split']).to_json('data/usuarios_backend.json', orient='records', indent=2, force_ascii=False)
df_transacciones.drop(columns=['split']).to_json('data/transacciones_backend.json', orient='records', indent=2, force_ascii=False)

print("¡Exportación exitosa! Semillas crudas y limpias generadas correctamente.")




¡Exportación exitosa! Semillas crudas y limpias generadas correctamente.


In [9]:
# check rápido
df_usuarios.head()

,id,nombre,ingreso_mensual,nivel_endeudamiento,frecuencia_ahorro,perfil_financiero,split
0,1,Feliciana,2353.28,15.98,Media,Saludable,val
1,2,Dani,4257.03,22.35,Alta,En observacion,train
2,3,Amador,3267.47,14.43,Baja,En observacion,train
3,4,Manuela,2879.85,14.35,Alta,Saludable,test
4,5,Leonardo,1925.49,5.45,Alta,Saludable,train
